# 16. Scaling and feature engineering
Scaling changes numerical representation. StandardScaler subtracts training means and divides by training standard deviations. MinMaxScaler uses the training range. RobustScaler uses robust summaries such as a median and interquartile range. Scaling does not require or force a feature to become normally distributed.

Use scaling when an algorithm is sensitive to distances or coefficient penalties. Tree splits often need no scaling. Feature engineering constructs useful inputs such as age at prediction time or a ratio. Every ingredient must be available at that time. Polynomial features can expand a feature space rapidly and require regularization and cross-validation.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
X = np.array([[1, 100], [2, 200], [3, 300]]) # Features with different units.
scaler = StandardScaler().fit(X) # Learn per-feature means and scales.
scaled = scaler.transform(X) # Apply the learned numerical conversion.
print("Means:", scaler.mean_) # Inspect original-unit averages.
print("Scaled means:", scaled.mean(axis=0).round(6)) # Training means are near zero.
poly = PolynomialFeatures(degree=2, include_bias=False) # Add squares and interactions.
print("Expanded names:", poly.fit(X).get_feature_names_out().tolist()) # Inspect added inputs.
assert np.allclose(scaled.mean(axis=0), 0) # Check this training transformation.

## Choose a transformation for a reason
RobustScaler centers by the median and scales by an interquartile range; outliers still exist afterward. PowerTransformer with Yeo–Johnson can transform positive, zero and negative values; Box–Cox requires strictly positive values. QuantileTransformer learns an empirical ranking map and can reshape marginals but distort distances and flatten unseen extremes at the learned bounds. KBinsDiscretizer replaces continuous values with bins; the strategy determines how boundaries are chosen. SplineTransformer constructs smooth piecewise-polynomial basis features. More sophisticated transformations add assumptions and parameters, so evaluate them in complete pipelines.

In [ ]:
import numpy as np
from sklearn.preprocessing import RobustScaler, PowerTransformer, QuantileTransformer, KBinsDiscretizer, SplineTransformer
X = np.array([[-3.], [-1.], [0.], [1.], [2.], [4.], [8.], [50.]]) # Include skew and a large extreme.
transformers = [RobustScaler(), PowerTransformer(method="yeo-johnson"), QuantileTransformer(n_quantiles=8, output_distribution="normal", random_state=42), KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform"), SplineTransformer(n_knots=3, degree=2, include_bias=False)] # Explicit bounded configurations.
for transformer in transformers:
    Z = transformer.fit_transform(X) # Illustrate the representation on this small training table.
    print(type(transformer).__name__, "shape", Z.shape, "first row", Z[0].round(3).tolist()) # Inspect width and scale.
    assert Z.shape[0] == len(X) and np.isfinite(Z).all() # Preserve examples and numerical validity.

### Reading the result
The first three mappings retain one input column; discretization assigns bin numbers and splines create multiple basis features. Ordinal bin numbers impose an order; one-hot bins are a different representation. These training transforms are not an evaluation. A transformed normal marginal does not make the joint data distribution normal.

### Transfer exercise
Compare a linear model with raw, power-transformed, and spline-expanded input using development cross-validation. Plot the mappings before interpreting a better score.

## Your turn
Compare a scaled and unscaled nearest-neighbor pipeline on identical folds. Keep feature engineering inside each evaluated pipeline.

Try the task before opening the solution on the course website.